In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
x_1=pd.read_csv(r"C:\Users\shubh\OneDrive\Desktop\datasets\new_x\merged_text_labels.csv")
x_1.head()

,id,text,label
0,731166399389962242,🔥ca kkk grand wizard 🔥 endorses @hillaryclinto...,unverified
1,714598641827246081,an open letter to trump voters from his top st...,unverified
2,691809004356501505,america is a nation of second chances —@potus ...,non-rumor
3,693204708933160960,"brandon marshall visits and offers advice, sup...",non-rumor
4,551099691702956032,rip elly may clampett: so sad to learn #beverl...,true


In [7]:
x_2=pd.read_csv(r"C:\Users\shubh\OneDrive\Desktop\datasets\new_x\merged_text_labels_v2.csv")
x_2.head()

,id,text,label
0,656955120626880512,correct predictions in back to the future ii URL,false
1,615689290706595840,.@whitehouse in rainbow colors for #scotusmarr...,true
2,613404935003217920,cops bought the alleged church shooter burger ...,false
3,731166399389962242,🔥ca kkk grand wizard 🔥 endorses @hillaryclinto...,unverified
4,714598641827246081,an open letter to trump voters from his top st...,unverified


In [11]:
x=pd.concat((x_1,x_2),axis=0,ignore_index=True)
x

,id,text,label
0,731166399389962242,🔥ca kkk grand wizard 🔥 endorses @hillaryclinto...,unverified
1,714598641827246081,an open letter to trump voters from his top st...,unverified
2,691809004356501505,america is a nation of second chances —@potus ...,non-rumor
3,693204708933160960,"brandon marshall visits and offers advice, sup...",non-rumor
4,551099691702956032,rip elly may clampett: so sad to learn #beverl...,true
...,...,...,...
2303,693546915892428800,jeb bush campaign kicks off 3-state farewell t...,non-rumor
2304,544269749405097984,breaking: live coverage of hostage situation u...,true
2305,760109079133990912,“after school satan clubs”? URL,unverified
2306,779633844680962048,this network of tunnels is from the stone age ...,unverified


In [13]:
x.isnull().sum()

id       0
text     0
label    0
dtype: int64

In [15]:
x.drop("id",axis=1,inplace=True)

In [19]:
x["label"].value_counts()

label
non-rumor     579
true          579
unverified    575
false         575
Name: count, dtype: int64

In [21]:
from sklearn.preprocessing import LabelEncoder

# Take labels
y = x['label']

# Initialize encoder
le = LabelEncoder()

# Fit and transform
y_encoded = le.fit_transform(y)

# Show mapping
print("Label classes:", le.classes_)
print("Example encoded labels:", y_encoded[:10])


Label classes: ['false' 'non-rumor' 'true' 'unverified']
Example encoded labels: [3 3 1 1 2 1 3 2 3 0]


In [27]:
X = x['text']

In [59]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import string

# ----------------------------
# Text Cleaning Function
# ----------------------------
def clean_text(text):
    # Remove URLs
    text = re.sub(r'http\S+|www.\S+', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Lowercase
    text = text.lower()
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
X_cleaned = X.apply(clean_text)
y = y_encoded 

# ----------------------------
# Train-test split (with cleaned text)
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned, y, test_size=0.2, random_state=42, stratify=y
)

# ----------------------------
# TF-IDF Vectorization (better tuned)
# ----------------------------
tfidf = TfidfVectorizer(
    max_features=10000,       # more features
    ngram_range=(1,3),        # unigrams, bigrams, trigrams
    stop_words='english'      # remove common words like "the", "is"
)

X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)
print("Sample labels (y_train):", y_train[:10])


TF-IDF train shape: (1846, 10000)
TF-IDF test shape: (462, 10000)
Sample labels (y_train): [1 1 2 0 2 0 1 1 2 3]


In [79]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers

# ----------------------------
# Build Deep Learning Model
# ----------------------------
model = keras.Sequential([
    layers.Input(shape=(X_train_tfidf.shape[1],)),   # input size = TF-IDF features
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.BatchNormalization(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.4),
    layers.BatchNormalization(),
    layers.Dense(len(np.unique(y)), activation='softmax')  # 4 classes
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Model summary
model.summary()


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_35 (Dense)                     │ (None, 128)                 │       1,280,128 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_26 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_36 (Dense)                     │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_27 (Dropout)                 │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_37 (Dense)                     │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_28 (Dropout)                 │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 32)                  │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_38 (Dense)                     │ (None, 4)                   │             132 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,291,236 (4.93 MB)

 Trainable params: 1,290,916 (4.92 MB)

 Non-trainable params: 320 (1.25 KB)

In [81]:
history = model.fit(
    X_train_tfidf, y_train,
    validation_data=(X_test_tfidf, y_test),
    epochs=10,          # you can try 15–20 for better learning
    batch_size=32,
    verbose=1
)


Epoch 1/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.2973 - loss: 1.7640 - val_accuracy: 0.2532 - val_loss: 1.3796
Epoch 2/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4091 - loss: 1.3831 - val_accuracy: 0.2511 - val_loss: 1.3648
Epoch 3/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.4956 - loss: 1.1587 - val_accuracy: 0.2511 - val_loss: 1.3363
Epoch 4/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6269 - loss: 0.9170 - val_accuracy: 0.3009 - val_loss: 1.2643
Epoch 5/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7139 - loss: 0.7469 - val_accuracy: 0.4740 - val_loss: 1.1452
Epoch 6/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7905 - loss: 0.6010 - val_accuracy: 0.6645 - val_loss: 0.9999
Epoch 7/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8531 - loss: 0.4759 - val_accuracy: 0.7684 - val_loss: 0.8488
Epoch 8/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8638 - loss: 0.3976 - val_accuracy: 0.8160 - v

In [83]:
# Save Keras model
model.save("rumor_classifier_model.h5")


In [87]:
import pickle

# Save vectorizer
with open("rumor_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
